# Descarga y preparación de datos agrícolas

## Proyecto Agronomía

Esta libreta documenta la obtención y preparación de los datos agrícolas
utilizados para estudiar los factores asociados con el éxito de la
producción agrícola en los municipios de México entre **2003 y 2025**.

Los archivos agrícolas se obtienen anualmente de la base de datos abiertos de
la Secretaría de Agricultura y posteriormente se integran en un único
conjunto histórico.

Debido a que la estructura de los archivos presenta algunos cambios a
lo largo del periodo de estudio, el procesamiento incluye una etapa de
normalización para mantener un esquema consistente entre los años.

La lógica de descarga y transformación se encuentra implementada en el
paquete `proyecto_agronomia`. Esta libreta permite ejecutar, inspeccionar
y validar el proceso de manera reproducible.

### Periodo de estudio

**2003–2025**

### Flujo de trabajo

1. Descargar los archivos agrícolas anuales.
2. Verificar la disponibilidad de los archivos.
3. Inspeccionar un archivo agrícola.
4. Normalizar las diferencias históricas del esquema.
5. Construir el conjunto agrícola histórico.
6. Generar variables derivadas e indicadores de calidad.
7. Validar el conjunto resultante.

In [ ]:
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIO_INICIAL,
    ANIO_FINAL,
    ANIOS,
)

from proyecto_agronomia.dataset import (
    descargar_datos_agricolas,
)

from proyecto_agronomia.features import (
    cargar_archivo_agricola,
    construir_historico_agricola,
    generar_features_agricolas,
)

## 1. Descarga de los datos agrícolas

La fuente agrícola proporciona un archivo independiente para cada año
del periodo de estudio.

Para construir la serie histórica se consideran los años comprendidos
entre **2003 y 2025**.

Los archivos originales se almacenan en:

`data/raw/agricultura/`

Los datos de esta carpeta representan la información original obtenida
de la fuente y no son modificados directamente durante las etapas
posteriores del procesamiento.

La función de descarga verifica si cada archivo ya existe localmente.
Cuando un archivo está disponible, se reutiliza en lugar de volver a
descargarlo.

In [3]:
print(
    f"Periodo de estudio: "
    f"{ANIO_INICIAL}-{ANIO_FINAL}"
)

print(
    f"Años considerados: "
    f"{len(ANIOS)}"
)

print(
    f"Directorio de datos originales: "
    f"{AGRICULTURA_RAW_DIR}"
)

Periodo de estudio: 2003-2025
Años considerados: 23
Directorio de datos originales: /home/dusty/Documents/Repositories/proyecto_agronomia/data/raw/agricultura


## 2. Ejecución de la descarga

Se utiliza la función `descargar_datos_agricolas()` del módulo
`proyecto_agronomia.dataset`.

La función recorre los años del periodo de estudio y conserva los
archivos que ya se encuentran disponibles localmente, por lo que la
operación puede ejecutarse nuevamente sin descargar innecesariamente
los mismos datos.

In [4]:
archivos_agricolas = descargar_datos_agricolas(
    ANIOS
)

print(
    f"Archivos disponibles: "
    f"{len(archivos_agricolas)}"
)

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

2026-11-14 11:10:36.192 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2003] Descargando...


Datos agrícolas:   4%|▍         | 1/23 [00:02<00:43,  1.99s/it]

2026-11-14 11:10:38.178 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2003] Descargado (5.80 MB)
2026-11-14 11:10:38.192 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2004] Descargando...


Datos agrícolas:   9%|▊         | 2/23 [00:08<01:35,  4.55s/it]

2026-11-14 11:10:44.512 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2004] Descargado (5.89 MB)
2026-11-14 11:10:44.521 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2005] Descargando...


... (38 outputs left out)

Datos agrícolas:  96%|█████████▌| 22/23 [01:23<00:04,  4.50s/it]

2026-11-14 11:11:59.704 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2024] Descargado (5.68 MB)
2026-11-14 11:11:59.712 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:51 - [2025] Descargando...


Datos agrícolas: 100%|██████████| 23/23 [01:24<00:00,  3.69s/it]

2026-11-14 11:12:01.026 | SUCCESS  | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2025] Descargado (5.50 MB)
Archivos disponibles: 23


## 3. Verificación de los archivos

Antes de iniciar el procesamiento se verifica que exista un archivo
para cada uno de los años considerados en el estudio.

Esta validación permite detectar años faltantes antes de construir el
conjunto histórico. Por defecto esta de 2003 a 2025, entonces esperamos 23 archivos

In [10]:
disponibilidad = pd.DataFrame({
        "Anio": ANIOS,
        "Archivo correspondiente": [
            f"agricola_{anio}.csv"
            for anio in ANIOS if (AGRICULTURA_RAW_DIR / f"agricola_{anio}.csv").exists()
        ]
    }
)

disponibilidad

,Anio,Archivo correspondiente
0,2003,agricola_2003.csv
1,2004,agricola_2004.csv
2,2005,agricola_2005.csv
3,2006,agricola_2006.csv
4,2007,agricola_2007.csv
5,2008,agricola_2008.csv
6,2009,agricola_2009.csv
7,2010,agricola_2010.csv
8,2011,agricola_2011.csv
9,2012,agricola_2012.csv


In [13]:
print(
    f"Archivos esperados: "
    f"{len(ANIOS)}"
)

print(
    f"Archivos disponibles: "
    f"{disponibilidad.shape[0]}"
)

print(
    f"Archivos faltantes: "
    f"{ ANIO_FINAL - ANIO_INICIAL + 1 - disponibilidad.shape[0]}"
)

Archivos esperados: 23
Archivos disponibles: 23
Archivos faltantes: 0


## 4. Inspección de los datos agrícolas

Cada archivo contiene registros agrícolas asociados con una entidad,
municipio, cultivo, ciclo productivo y modalidad.

Entre las principales variables disponibles se encuentran:

- superficie sembrada;
- superficie cosechada;
- superficie siniestrada;
- volumen de producción;
- rendimiento;
- precio medio rural;
- valor de la producción.

A continuación se inspecciona uno de los archivos del periodo de
estudio antes de construir el conjunto histórico.

In [15]:
ruta_ejemplo = (AGRICULTURA_RAW_DIR / "agricola_2025.csv")

agricultura_ejemplo = cargar_archivo_agricola(ruta_ejemplo, 2025,)

agricultura_ejemplo.head()

2026-11-14 12:09:52.045 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2025] Columnas normalizadas.


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


In [16]:
print(
    f"Observaciones: "
    f"{len(agricultura_ejemplo):,}"
)

print(
    f"Variables: "
    f"{agricultura_ejemplo.shape[1]}"
)

print(
    f"Estados: "
    f"{agricultura_ejemplo['Idestado'].nunique()}"
)

print(
    f"Municipios: "
    f"{agricultura_ejemplo[['Idestado', 'Idmunicipio']].drop_duplicates().shape[0]:,}"
)

print(
    f"Cultivos: "
    f"{agricultura_ejemplo['Nomcultivo'].nunique():,}"
)

Observaciones: 35,902
Variables: 24
Estados: 32
Municipios: 2,441
Cultivos: 314


## 5. Normalización histórica

Los archivos agrícolas no mantienen exactamente el mismo esquema de
columnas durante todo el periodo 2003–2025.

Se identificaron cambios históricos en variables como el nombre del
cultivo y el precio medio rural.

Para construir una serie consistente, el pipeline normaliza estas
diferencias antes de concatenar los archivos anuales.

Entre los cambios considerados se encuentran:

- `Nomcultivo Sin Um` → `Nomcultivo`
- `Precio` → `Preciomediorural`

También se convierten las principales variables cuantitativas a formato
numérico y se eliminan separadores de miles cuando están presentes.

Los archivos originales permanecen sin modificaciones en
`data/raw/agricultura/`; la normalización se realiza durante la
construcción del conjunto histórico.

## 6. Construcción del conjunto agrícola histórico

Una vez disponibles y normalizados los archivos anuales, se integran
los datos correspondientes al periodo **2003–2025** en un único
conjunto histórico.

Durante este proceso:

1. Se carga cada archivo anual.
2. Se normalizan las diferencias de nombres entre años.
3. Se convierten las principales variables cuantitativas a formato
   numérico.
4. Se concatenan todos los registros conservando el año y las
   características originales de cada observación.

Esta integración permite trabajar con todo el periodo de estudio bajo
un esquema común.

In [17]:
agricultura_historica = construir_historico_agricola()

agricultura_historica.shape

Procesando datos agrícolas:   4%|▍         | 1/23 [00:00<00:07,  3.10it/s]

2026-11-14 12:10:03.816 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2003] Columnas normalizadas.
2026-11-14 12:10:04.014 | INFO     | proyecto_agronomia.features:construir_historico_agricola:147 - [2003] 35,667 registros procesados.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:00<00:07,  3.10it/s]

2026-11-14 12:10:04.085 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2004] Columnas normalizadas.


Procesando datos agrícolas:   9%|▊         | 2/23 [00:00<00:06,  3.27it/s]

2026-11-14 12:10:04.308 | INFO     | proyecto_agronomia.features:construir_historico_agricola:147 - [2004] 36,204 registros procesados.
2026-11-14 12:10:04.413 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2005] Columnas normalizadas.


... (39 outputs left out)

2026-11-14 12:10:10.774 | INFO     | proyecto_agronomia.features:construir_historico_agricola:147 - [2024] 36,992 registros procesados.
2026-11-14 12:10:10.908 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2025] Columnas normalizadas.


Procesando datos agrícolas: 100%|██████████| 23/23 [00:07<00:00,  3.08it/s]

2026-11-14 12:10:11.165 | INFO     | proyecto_agronomia.features:construir_historico_agricola:147 - [2025] 35,902 registros procesados.


(825089, 24)

In [18]:
agricultura_historica.head()

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
0,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1062.0,1062.0,0.0,28529.0,26.86,323.4,9226278.6
1,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5900000,Cebada forrajera en verde,2.0,2.0,0.0,54.0,27.00,250.0,13500.0
2,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5980000,Centeno forrajero en verde,17.0,17.0,0.0,1020.0,60.00,350.0,357000.0
3,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,9090000,Triticale forrajero en verde,24.0,24.0,0.0,720.0,30.00,400.0,288000.0
4,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,15050000,Pastos y praderas,415.0,415.0,0.0,25830.0,62.24,314.9,8133867.0


In [ ]:
print(
    f"Observaciones totales: "
    f"{len(agricultura_historica):,}"
)

print(
    f"Variables: "
    f"{agricultura_historica.shape[1]}"
)

print(
    f"Periodo: "
    f"{agricultura_historica['Anio'].min()}-"
    f"{agricultura_historica['Anio'].max()}"
)

print(
    f"Años disponibles: "
    f"{agricultura_historica['Anio'].nunique()}"
)

## 7. Cobertura temporal

Se verifica el número de registros disponibles para cada año del
periodo de estudio.

Esta revisión permite comprobar que los 23 años están representados y
detectar cambios importantes en la cantidad de observaciones entre
periodos.

In [ ]:
registros_por_anio = (
    agricultura_historica
    .groupby("Anio")
    .size()
    .reset_index(name="Registros")
)

registros_por_anio

## 8. Revisión de calidad de los datos

Antes de generar variables derivadas se realizan comprobaciones básicas
sobre las variables agrícolas cuantitativas.

Se revisan:

- valores faltantes;
- registros duplicados;
- valores negativos;
- superficie sembrada igual a cero;
- superficie cosechada mayor que la sembrada;
- balance entre superficie cosechada, siniestrada y sembrada.

Las observaciones potencialmente inconsistentes no se eliminan
automáticamente. En etapas posteriores se generan indicadores que
permiten identificarlas y conservar el dato original.

In [ ]:
columnas_numericas = [
    "Sembrada",
    "Cosechada",
    "Siniestrada",
    "Volumenproduccion",
    "Rendimiento",
    "Preciomediorural",
    "Valorproduccion",
]

faltantes = (
    agricultura_historica[columnas_numericas]
    .isna()
    .sum()
    .to_frame("Faltantes")
)

faltantes

In [ ]:
duplicados_exactos = (
    agricultura_historica
    .duplicated()
    .sum()
)

print(
    f"Registros duplicados exactos: "
    f"{duplicados_exactos:,}"
)

In [ ]:
negativos = (
    agricultura_historica[columnas_numericas]
    .lt(0)
    .sum()
    .to_frame("Valores_negativos")
)

negativos

## 9. Generación de variables derivadas

A partir de las variables agrícolas originales se genera
`TasaCosecha`, definida como:

$$
\text{TasaCosecha} =
\frac{\text{Cosechada}}{\text{Sembrada}}
$$

Esta variable representa la proporción de la superficie sembrada que
llegó a ser cosechada.

Cuando `Sembrada = 0`, la tasa se mantiene como valor faltante para
evitar una división indefinida.

También se generan indicadores de calidad que permiten identificar
observaciones potencialmente inconsistentes sin modificar ni eliminar
los registros originales:

- `FlagSembradaCero`
- `FlagCosechadaMayorSembrada`
- `FlagBalanceSuperficie`

In [ ]:
agricultura_features = generar_features_agricolas(
    agricultura_historica
)

agricultura_features.head()

In [ ]:
agricultura_features.shape

In [ ]:
print(
    "TasaCosecha faltante:",
    agricultura_features[
        "TasaCosecha"
    ].isna().sum()
)

print(
    "Sembrada = 0:",
    agricultura_features[
        "FlagSembradaCero"
    ].sum()
)

print(
    "Cosechada > Sembrada:",
    agricultura_features[
        "FlagCosechadaMayorSembrada"
    ].sum()
)

print(
    "Cosechada + Siniestrada > Sembrada:",
    agricultura_features[
        "FlagBalanceSuperficie"
    ].sum()
)

In [ ]:
tasa_mayor_uno = (
    agricultura_features[
        agricultura_features["TasaCosecha"] > 1
    ][
        [
            "Anio",
            "Nomestado",
            "Nommunicipio",
            "Nomcultivo",
            "Sembrada",
            "Cosechada",
            "TasaCosecha",
        ]
    ]
)

tasa_mayor_uno

## Resultado

Se construyó un conjunto agrícola histórico homogéneo para el periodo
2003–2025.

El conjunto conserva los registros originales y agrega variables
derivadas necesarias para las siguientes etapas del análisis.

La variable `TasaCosecha` permitirá estudiar la proporción de superficie
sembrada que llega a cosecharse y posteriormente analizar su relación
con factores agrícolas, geográficos y climáticos.

Las observaciones potencialmente inconsistentes se conservaron y fueron
identificadas mediante indicadores de calidad, evitando realizar
correcciones o eliminaciones automáticas sin una justificación
metodológica.

El conjunto resultante queda preparado para su posterior integración
con los datos climáticos mediante las variables de año, estado y
municipio.